<a href="https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/CIS_5450_Project_Difficulty_Topics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIS 5450 Project: Difficulty Topics
**Group Members:**
* **Shangyi Du**
* **Jingyi Gong**
* **Chenning Huang**


## Topic 1: Entity Linking
[Hyperlink](https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/ufo_sightings.ipynb#scrollTo=mH3thvhSHusP&line=1&uniqifier=1)

### Why we used this concept
We applied entity linking because our research aims to examine whether contextual factors beyond direct observations, such as environmental conditions, geographic context, and cultural or socioeconomic characteristics, may influence how UFO shapes are reported. The NUFORC sightings dataset alone lacks these additional predictors, so we incorporated multiple external datasets at the city and country levels. Since these contextual features are not directly aligned to sighting coordinates, entity linking was necessary to accurately assign the nearest valid environmental and human-contextual attributes to each UFO report. This enriched dataset provides a more comprehensive feature space that enables us to explore a wider range of potential influences in subsequent modeling and analysis.

### How we implemented it
We applied two entity linking:

1. **UFO Sightings → City Features (Nearest-Valid City Matching)**  
   In the city features dataset, `population`, `avg_annual_temp(°C)`, `temp_seasonality`, `annual_precipitation(mm)`, `elevation(m)`, and `viirs_annual_ave` contain missing values for some cities. If we simply merged each UFO sighting with the single nearest city, some UFO records would inherit missing feature values even though a slightly more distant nearby city has complete data.

   To address this, we implemented a nearest-non-null matching procedure:

   - For each UFO sighting, we identified the k nearest cities using BallTree K-nearest-neighbor search in projected coordinates.

   - For each feature, we examined these candidate cities in order of distance (nearest → farther).

   - We selected the closest city that actually has a valid (non-null) value for that feature.

   - We also recorded the distance to the city that supplied the data, producing a meaningful measure of spatial accuracy.


2. **UFO Sightings → Country Education Data (Reverse Geocoding)**  
   Because education indicators are reported at the country level, we:

   - Cleaned and numeric-converted the latitude/longitude fields.

   - Reverse-geocoded coordinates into standardized ISO-aligned country labels.

   - Merged the dataset via a normalized "country" key.


### Results & Interpretation
Entity linking allowed us to successfully augment each UFO sighting with environmental, geographic, and human-contextual features while avoiding missing-value propagation. By assigning the closest valid city for each contextual variable, this approach preserves geographic proximity and ensures that we do not introduce incomplete or unrealistic data. It also standardizes national attributes through reverse geocoding, associating each sighting with consistent country-level cultural and socioeconomic indicators. As a result, we obtained a more complete and robust feature set—one that maintains spatial realism, resolves missing data issues, and provides a reliable foundation for downstream modeling and analysis.

## Topic 2: Imbalance data (SMOTE)
[Hyperlink](https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/ufo_sightings.ipynb#scrollTo=kkAI_oyt5zE8)
### Why we used this concept
Our target variable has 21 categories with a highly imbalanced distribution. A few shapes (especially "light" and "triangle") account for a large fraction of the sightings, while many other shapes (e.g., "cone", "cross", "teardrop") are rare.

When we trained models on the original data, we observed that:


*   The majority-class baseline already achieved 21% accuracy by always predicting "light".
*   More complex models tended to focus on the dominant classes and almost ignore the rare ones, leading to low macro-style F1 scores.

To address this, we implemented SMOTE (Synthetic Minority Oversampling TEchnique). This allows us to:

*   Give each shape class a sufficient number of training examples.
*   Encourage models to discover more patterns in minority classes.





### How we implemented it
We applied SMOTE on the **training set** after splitting the data. Using the SMOTE function from imblearn, we generated synthetic samples for the minority classes so that each shape category had a more balanced number of examples. After resampling, we re-encoded the target labels with One-Hot encoding to make the balanced dataset compatible with our multi-class models.


### Results & Interpretation
SMOTE had a very clear effect on training metrics, but only a limited effect on test performance.

For tree-based models (e.g. Random Forest and AdaBoost), SMOTE dramatically increased training accuracy and weighted macro F1 (even above 0.9), showing that the models can easily fit the oversampled data, while the test accuracy improves only slightly which implies overfitting which is indeed the main limitation of tree-based models.

XGBoost benefited the most from SMOTE. With SMOTE, XGBoost's test accuracy rose to about 16.4% and weighted F1 improved to 0.125, which is still below the 21% majority baseline but better than the original XGBoost performance.

For SVM, SMOTE improved training metrics on the subsampled training set, but test accuracy remained around 4 - 5% with very low weighted macro F1, indicating that the SVM still struggles in this high-dimensional, noisy, multi-class setting.

Overall, SMOTE successfully balanced the training distribution and made minority classes more visible during training. However, it did not fully solve the generalization problem.

This suggests that, for our UFO shape prediction task, the main limitation is the weak signal in the current features rather than the imbalance technique itself. SMOTE is helpful for exploration, but it cannot create class separability where there is very little information to begin with.


## Topic 3: Visualization Packages (plotly, folium)
[Plotly](https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/ufo_sightings.ipynb#scrollTo=WdCB8j8ng-JU)

[Folium](https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/ufo_sightings.ipynb#scrollTo=K0sbZyKE1f6U)
### Why we used this concept

Heatmaps are an effective visualization technique for representing the magnitude of values across two dimensions using color intensity.

* Geographic heatmap (Folium HeatMap): To visualize the spatial density of UFO sightings across the globe. This approach was chosen because it allows viewers to immediately identify geographic clusters and sparse regions without needing to interpret thousands of individual data points. The human eye naturally perceives color gradients, making it easier to spot patterns like the concentration of sightings in North America versus the scarcity in other continents.

* Categorical heatmap (Plotly): To display the normalized distribution of UFO shapes across 131 countries. This was necessary because comparing raw counts across countries with vastly different reporting volumes would be misleading, and there are too many countries in the plot, which leads to the not complete showing the label of countries in the y axis. By using Plotly's interactive visualization, viewers can hover over individual cells to identify specific country-shape combinations of interest, particularly those with dominant proportions indicated by brighter colors, without relying on cramped axis labels.

We used these techniques because we'd like to explore spatial and proportional visualizations, which directly connect to the project goal: identifying how contextual (geographic, cultural, infrastructural) factors influence UFO reporting.

---

### How we implemented it

* Geographic Heatmap (Section 3.1.1):

  A Folium HeatMap was created by assigning an equal weight to each sighting (intensity = 1) to treat all reports equally. We extracted latitude, longitude, and intensity as input data for the heatmap layer. To enhance color contrast, we utilized a Folium map with dark tiles (CartoDB dark_matter) and applied a custom color gradient progressing from blue (low density) to red (high density).

* Categorical Heatmap (Section 3.1.3):

  We grouped sightings by country and shape, then counted occurrences, normalized counts within each country by dividing by the country's total sightings. Then we pivoted the data into a matrix format with countries as rows and shapes as columns, and visualized using Plotly's imshow with the Viridis color scale for perceptual uniformity across countries.


### Results & Interpretation

In 3.1.1, the geographic heatmap allowed us to rapidly assess the spatial distribution of UFO sightings without manually aggregating counts by region. By visualizing density through color intensity, we immediately identified North America and Western Europe as reporting hotspots while confirming sparse coverage across Africa and South America. This insight revealed that the dataset has a bias in countries in number of reporting, informing our subsequent steps using country as a feature to reflect if dataset reflects reporting infrastructure and internet accessibility rather than a uniform global phenomenon.

Moreover, the categorical heatmap in 3.1.3 enabled efficient exploration of shape distributions across 131 countries, which is a high-dimensional space that would be impractical to examine through tabular summaries alone. The interactive hover functionality surfaced outliers such as Fiji's concentration of "oval" sightings and Oman's dominance of "cigar" shapes, generating hypotheses about cultural or environmental factors without requiring exhaustive statistical testing. Normalization ensured that countries with vastly different reporting volumes could be compared on equal footing, revealing that shape preferences vary meaningfully across nations. As a result, we obtained visual confirmation that country-level features may carry predictive information about shape classification, providing a reliable foundation for downstream modeling decisions.